In [5]:
# """Question 6: Use the following small text dataset to train a simple Variational Autoencoder (VAE) for text reconstruction: 
# ["The sky is blue", "The sun is bright", "The grass is green", 
# "The night is dark", "The stars are shining"] 
# 1. Preprocess the data (tokenize and pad the sequences). 
# 2. Build a basic VAE model for text reconstruction. 
# 3. Train the model and show how it reconstructs or generates similar sentences. Include your code, explanation, and sample outputs. 
# (Include your Python code and output in the code box below.)"""
# # Answer:- 
# # 1. Data Preprocessing
# # Tokenize sentences
# # Convert to sequences
# # Pad sequences
# # 2. Build VAE Model
# # Encoder → latent mean & variance
# # Sampling layer
# # Decoder → reconstruct sentence
# # 3. Python Code + Output
# ==============================
# 1. Imports
# ==============================
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

# ==============================
# 2. Dataset
# ==============================
texts = [
    "The sky is blue",
    "The sun is bright",
    "The grass is green",
    "The night is dark",
    "The stars are shining"
]

# ==============================
# 3. Preprocessing
# ==============================
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

max_len = max(len(seq) for seq in sequences)
vocab_size = len(tokenizer.word_index) + 1

padded = pad_sequences(sequences, maxlen=max_len, padding='post')

# ==============================
# 4. Encoder
# ==============================
embedding_dim = 16
latent_dim = 8

encoder_inputs = Input(shape=(max_len,))
x = Embedding(vocab_size, embedding_dim)(encoder_inputs)
x = LSTM(32)(x)

z_mean = Dense(latent_dim)(x)
z_log_var = Dense(latent_dim)(x)

# Sampling
from tensorflow.keras.layers import Lambda

def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = Lambda(sampling)([z_mean, z_log_var])

encoder = Model(encoder_inputs, [z_mean, z_log_var, z])

# ==============================
# 5. Decoder
# ==============================
from tensorflow.keras.layers import Reshape

decoder_inputs = Input(shape=(latent_dim,))
x = Dense(32, activation='relu')(decoder_inputs)
x = Dense(max_len * vocab_size)(x)

# ✅ FIXED
x = Reshape((max_len, vocab_size))(x)
decoder_outputs = tf.keras.layers.Activation('softmax')(x)

decoder = Model(decoder_inputs, decoder_outputs)

# ==============================
# 6. VAE Model (Subclass)
# ==============================
class VAE(Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            # Reconstruction loss
            recon_loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(data, reconstruction)
            )

            # KL loss
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )

            total_loss = recon_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        return {"loss": total_loss}

# ==============================
# 7. Train
# ==============================
vae = VAE(encoder, decoder)
vae.compile(optimizer='adam')

vae.fit(padded, epochs=50, verbose=1)

# ==============================
# 8. Reconstruction
# ==============================
z_mean, z_log_var, z = encoder.predict(padded)
preds = decoder.predict(z)

index_word = {v: k for k, v in tokenizer.word_index.items()}

def decode_sequence(seq):
    words = []
    for token_probs in seq:
        idx = np.argmax(token_probs)
        words.append(index_word.get(idx, ""))
    return " ".join(words)

print("\nReconstructed Sentences:\n")
for pred in preds:
    print(decode_sequence(pred))

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - loss: 0.0000e+00
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.0000e+00
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0000e+00
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.0000e+00
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0000e+00
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.0000e+00
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - loss: 0.0000e+00
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0000e+00
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 0.0000e+00
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0000e+00
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.0000e+00
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.0000e+00
Epoch 13/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0000e+00
Epoch 14/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0000e+00
Epoch 15/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/

In [6]:
# Question 7: Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short English paragraph into French and German. Provide the original and translated text. 
# (Include your Python code and output in the code box below.) 
# Answer:
from transformers import pipeline

# Load text-generation pipeline (GPT-2)
generator = pipeline("text-generation", model="gpt2")

# Input paragraph
text = "Artificial Intelligence is transforming the world by enabling machines to learn and make decisions."

# Prompt for translation
prompt = f"Translate the following English text to French and German:\n{text}\nFrench:"

# Generate output
result = generator(prompt, max_length=100, num_return_sequences=1)

print(result[0]['generated_text'])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7286.31it/s]
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Translate the following English text to French and German:
Artificial Intelligence is transforming the world by enabling machines to learn and make decisions.
French:
Artificial intelligence is transforming the world by enabling machines to learn and make decisions. German:
Artificial intelligence is transforming the world by enabling machines to learn and make decisions.
Artificial Intelligence is transforming the world by enabling machines to learn and make decisions.
The goal of artificial intelligence is to overcome the limitations of human nature and to transcend our cognitive biases.
Artificial Intelligence is transforming the world by enabling machines to learn and make decisions.
Artificial Intelligence is transforming the world by enabling machines to learn and make decisions.
Artificial Intelligence is transforming the world by enabling machines to learn and make decisions.
Artificial Intelligence is transforming the world by enabling machines to learn and make decisions.
Art

In [7]:
# Question 8: Implement a simple attention-based encoder-decoder model for English-to-Spanish translation using Tensorflow or PyTorch.
# Answer:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# ==============================
# 1. Sample Dataset
# ==============================
eng_sentences = ["hello", "how are you", "i am fine"]
spa_sentences = ["hola", "como estas", "estoy bien"]

# Tokenization
eng_tokenizer = Tokenizer()
spa_tokenizer = Tokenizer()

eng_tokenizer.fit_on_texts(eng_sentences)
spa_tokenizer.fit_on_texts(spa_sentences)

eng_seq = eng_tokenizer.texts_to_sequences(eng_sentences)
spa_seq = spa_tokenizer.texts_to_sequences(spa_sentences)

max_len_eng = max(len(seq) for seq in eng_seq)
max_len_spa = max(len(seq) for seq in spa_seq)

eng_seq = pad_sequences(eng_seq, maxlen=max_len_eng, padding='post')
spa_seq = pad_sequences(spa_seq, maxlen=max_len_spa, padding='post')

eng_vocab = len(eng_tokenizer.word_index) + 1
spa_vocab = len(spa_tokenizer.word_index) + 1

# ==============================
# 2. Encoder
# ==============================
embedding_dim = 32
units = 64

encoder_inputs = tf.keras.Input(shape=(max_len_eng,))
enc_emb = Embedding(eng_vocab, embedding_dim)(encoder_inputs)
encoder_outputs, state_h, state_c = LSTM(units, return_sequences=True, return_state=True)(enc_emb)

# ==============================
# 3. Attention Mechanism
# ==============================
attention = tf.keras.layers.Attention()
# Query = decoder, Value = encoder
# Will be used inside decoder

# ==============================
# 4. Decoder
# ==============================
decoder_inputs = tf.keras.Input(shape=(max_len_spa,))
dec_emb = Embedding(spa_vocab, embedding_dim)(decoder_inputs)

decoder_lstm = LSTM(units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

# Apply Attention
context = attention([decoder_outputs, encoder_outputs])

# Concatenate context + decoder output
concat = tf.keras.layers.Concatenate(axis=-1)([decoder_outputs, context])

# Final Dense layer
outputs = Dense(spa_vocab, activation='softmax')(concat)

# ==============================
# 5. Model
# ==============================
model = tf.keras.Model([encoder_inputs, decoder_inputs], outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ==============================
# 6. Training
# ==============================
model.fit([eng_seq, spa_seq], spa_seq, epochs=100, verbose=1)

# ==============================
# 7. Testing
# ==============================
preds = model.predict([eng_seq, spa_seq])

index_word = {v: k for k, v in spa_tokenizer.word_index.items()}

def decode(seq):
    words = []
    for token in seq:
        idx = np.argmax(token)
        words.append(index_word.get(idx, ""))
    return " ".join(words)

print("\nTranslations:\n")
for pred in preds:
    print(decode(pred))

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0000e+00 - loss: 1.7935
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.1667 - loss: 1.7900
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.1667 - loss: 1.7865
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5000 - loss: 1.7829
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8333 - loss: 1.7794
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 1.0000 - loss: 1.7757
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 1.0000 - loss: 1.7720
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 1.0000 - loss: 1.7681
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 1.0000 - loss: 1.7641
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 1.0000 - loss: 1.7599
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 1.0000 - loss: 1.7555
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 1.0000 - 

In [8]:
# Question 9: Use the following short poetry dataset to simulate poem generation with a pre-trained GPT model: 
# ["Roses are red, violets are blue,", 
# "Sugar is sweet, and so are you.", 
# "The moon glows bright in silent skies,", 
# "A bird sings where the soft wind sighs."] 
# Using this dataset as a reference for poetic structure and language, generate a new 2-4 line poem using a pre-trained GPT model (such as GPT-2). You may simulate fine-tuning by prompting the model with similar poetic patterns. 
# Include your code, the prompt used, and the generated poem in your answer. 
# (Include your Python code and output in the code box below.) 
# Answer:
from transformers import pipeline

# Load GPT-2
generator = pipeline("text-generation", model="gpt2")

# Dataset (used as style reference)
dataset = [
    "Roses are red, violets are blue,",
    "Sugar is sweet, and so are you.",
    "The moon glows bright in silent skies,",
    "A bird sings where the soft wind sighs."
]

# Prompt (simulate fine-tuning via pattern)
prompt = """Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
"""

# Generate poem
result = generator(prompt, max_length=60, num_return_sequences=1)

print("Generated Poem:\n")
print(result[0]['generated_text'])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9251.22it/s]
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Poem:

Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
And the stars are a kind of mystery.
The stars have been known as the stars of the universe.
The stars have shone out of thin air.
They have made a living, and yet they have been lost to us.
The stars are one, and we are not.
They are a part of the universe, a part of our existence.
The stars do not create us.
The stars do not make us.
The stars do not create a reality.
The stars are a kind of mystery.
The stars are a place.
The stars are a place of suffering.
The stars are a place for suffering.
The stars are a place for the world to find its way.
The stars are a place for the world to find its way out.
The stars are a place for the world to find its way back.
The stars are a place for the world to become a world of suffering.
The stars are a place for the world to become a world of suffering.
The stars are a place for the world to become a world of suffering.
The s

In [9]:
# # Question 10: Imagine you are building a creative writing assistant for a publishing company. The assistant should generate story plots and character descriptions using Generative AI. Describe how you would design the system, including model selection, training data, bias mitigation, and evaluation methods. Explain the real-world challenges you might face. 
# # Answer:
# 1. Model Selection
# Use Transformer-based models like Large Language Models
# Examples: GPT-4, LLaMA
# Fine-tune or prompt-engineer for:
# Story plots
# Character descriptions
# 2. Training Data
# Sources:
# Books, novels, scripts
# Public datasets (Project Gutenberg)
# Preprocessing:
# Clean text
# Remove duplicates
# Structure into:
# Plot summaries
# Character profiles
# 3. System Architecture
# Input: User prompt (genre, theme, character type)
# Processing:
# Prompt engineering / fine-tuned model
# Output:
# Story plot
# Character descriptions

# Optional:

# Add memory (user preferences)
# Add style control (fantasy, thriller, etc.)
# 4. Bias Mitigation
# Filter biased training data
# Use fairness checks on outputs
# Apply moderation layers to avoid:
# Stereotypes
# Offensive content
# 5. Evaluation Methods
# Automatic Metrics:
# Perplexity
# BLEU / ROUGE
# Human Evaluation:
# Creativity
# Coherence
# Originality
# 6. Real-World Challenges
#  Data Issues
# Copyright problems
# Limited high-quality datasets
#  Bias & Ethics
# Stereotypical characters
# Cultural bias
#  Consistency
# Maintaining long story coherence
#  Hallucination
# Unrealistic or illogical plots
#  Control
# Hard to enforce exact tone/style